--- ADIM 0 Excel Verilerini Python'a Aktarma.

In [12]:
from openpyxl import load_workbook
dosya = "sosyal_medya_500_kullanici_sade.xlsx"
excel = load_workbook(dosya)
print(excel.sheetnames)
kullanicilar_sayfasi = excel["Kullanicilar"] #Sayfaları Ayrı Değişkenlere Atadık.
haftalik_sayfasi = excel["HaftalikVeri"] #Sayfaları Ayrı Değişkenlere Atadık.
print("Kullanıcı sayısı:", kullanicilar_sayfasi.max_row - 1)
print("Haftalık veri sayısı:", haftalik_sayfasi.max_row - 1)

['Kullanicilar', 'HaftalikVeri']
Kullanıcı sayısı: 500
Haftalık veri sayısı: 4000


-- ADIM 1 Python Veri Yapılarını Karşılaştırma: LIST, TUPLE ve DICT 

In [13]:
#500 Kullanıcıyı 3 Farklı Yapıda Saklayıp Bellek Kullanımlarını Görelim.
# 1.LIST
kullanici_listesi = [] #Boş bir liste oluşturur.
for satir in kullanicilar_sayfasi.iter_rows(min_row=2, values_only=True): #Satır satır tek tek okur.
    kullanici_listesi.append(satir) #Her satırı listenin sonuna ekler.  
print(kullanici_listesi[:3]) #İlk 3 elemanı gösterir.

[('U0001', 100), ('U0002', 2098), ('U0003', 881)]


In [14]:
#2. TUPLE
kullanici_demeti = tuple(kullanici_listesi) 
print(kullanici_demeti[:3])

(('U0001', 100), ('U0002', 2098), ('U0003', 881))


In [15]:
# DICT
kullanici_sozlugu = {}
for satir in kullanici_listesi:
    kullanici_id = satir[0]
    takipci = satir[1]
    kullanici_sozlugu[kullanici_id] = takipci
print(kullanici_sozlugu["U0001"])

100


Şu Anda Veriler Böyle Duruyor:
    List: [("U0001", 100), ("U0002", 2098)] 
    Tuple: (("U0001", 100), ("U0002", 2098)) 
    Dict: {"U0001": 100, "U0002": 2098}

In [16]:
print("List:", len(kullanici_listesi))
print("Tuple:", len(kullanici_demeti))
print("Dict:", len(kullanici_sozlugu))

List: 500
Tuple: 500
Dict: 500


Deney 1: Bellek Kullanımı.

In [17]:
import sys

print("List:", sys.getsizeof(kullanici_listesi), "byte")
print("Tuple:", sys.getsizeof(kullanici_demeti), "byte")
print("Dict:", sys.getsizeof(kullanici_sozlugu), "byte")

List: 4216 byte
Tuple: 4040 byte
Dict: 13056 byte


In [18]:
# LIST
list_boyut = sys.getsizeof(kullanici_listesi)
for satir in kullanici_listesi:
    list_boyut += sys.getsizeof(satir)
    for veri in satir:
        list_boyut += sys.getsizeof(veri)

# TUPLE
tuple_boyut = sys.getsizeof(kullanici_demeti)
for satir in kullanici_demeti:
    tuple_boyut += sys.getsizeof(satir)
    for veri in satir:
        tuple_boyut += sys.getsizeof(veri)

# DICT
dict_boyut = sys.getsizeof(kullanici_sozlugu)
for kullanici_id, takipci in kullanici_sozlugu.items():
    dict_boyut += sys.getsizeof(kullanici_id)
    dict_boyut += sys.getsizeof(takipci)

# SONUÇLAR
print("List:", list_boyut, "byte")
print("Tuple:", tuple_boyut, "byte")
print("Dict:", dict_boyut, "byte")

List: 69216 byte
Tuple: 69040 byte
Dict: 50056 byte


In [19]:
# LIST ile arama
def list_ara():
    for satir in kullanici_listesi:
        if satir[0] == "U0450":
            return satir[1]

# TUPLE ile arama
def tuple_ara():
    for satir in kullanici_demeti:
        if satir[0] == "U0450":
            return satir[1]

# DICT ile arama
def dict_ara():
    return kullanici_sozlugu["U0450"]
    
print("List:", list_ara())
print("Tuple:", tuple_ara())
print("Dict:", dict_ara())

List: 308
Tuple: 308
Dict: 308


In [20]:
#Hızlarını Ölçelim. 
%timeit list_ara()
%timeit tuple_ara()
%timeit dict_ara()

19.8 μs ± 380 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
19.7 μs ± 472 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
80.2 ns ± 0.818 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


Deney 3: Veri Ekleme, Silme, Güncelleme İşlemleri.
Amacım: U0501 Kullanıcısını eklemek, U0450 takipcisini 999 yapmak ve U001 kullanıcısını silmek.

In [21]:
#List Kısmı
def list_islem():
    veri = kullanici_listesi.copy()
    veri.append(("U0501", 100)) #Veri Ekleme
    veri[449] = ("U0450", 999)  #Güncelleeme  
    veri.pop(0)                 #Silme   

In [22]:
#TUPLE Kısmı
#Tuple değiştirilemez. Bu yüzden mevcut üzerinde yapmak yerine yeni tuple oluşturmak zorundayız.
def tuple_islem():
    veri = kullanici_demeti
    veri = veri + (("U0501", 100),)   # Ekleme
    veri = veri[:449] + (("U0450", 999),) + veri[450:]  # Güncelleme
    veri = veri[1:]    # Silme

In [23]:
#DICT Kısmı
def dict_islem():
    veri = kullanici_sozlugu.copy()
    veri["U0501"] = 100   # Ekleme
    veri["U0450"] = 999   # Güncelleme
    del veri["U0001"]   # Silme

In [24]:
#Hız Testleri
%timeit list_islem()
%timeit tuple_islem()
%timeit dict_islem()

1.83 μs ± 54 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
7.2 μs ± 10.6 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
3.27 μs ± 66.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [28]:
import numpy as np
# Dict içerisindeki takipçi sayılarını alıyoruz.
takipciler = list(kullanici_sozlugu.values())
# Aynı verileri NumPy dizisine dönüştürüyoruz.
takipciler_np = np.array(takipciler)
print(type(takipciler))
print(type(takipciler_np))

<class 'list'>
<class 'numpy.ndarray'>


In [29]:
#Hesaplamaları Karşılaştırmak İçin:
# Temel Python
print("Toplam:", sum(takipciler))
print("Ortalama:", sum(takipciler) / len(takipciler))
print("Maksimum:", max(takipciler))

# NumPy
print("Toplam:", np.sum(takipciler_np))
print("Ortalama:", np.mean(takipciler_np))
print("Maksimum:", np.max(takipciler_np))

Toplam: 661916
Ortalama: 1323.832
Maksimum: 2496
Toplam: 661916
Ortalama: 1323.832
Maksimum: 2496


In [30]:
%timeit sum(takipciler)
%timeit np.sum(takipciler_np)

3.2 μs ± 59.3 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
2.99 μs ± 63.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [31]:
veriler = list(haftalik_sayfasi.values)[1:]
print("Toplam kayıt:", len(veriler))

Toplam kayıt: 4000


In [32]:
#Temel Python Kullanarak Hesaplama:
onceki = kullanici_sozlugu.copy()
toplam_degisim = [0] * 8
toplam_puan = [0] * 8
aktiflik_puanlari = {}
for id, hafta, post, begeni, yorum, takipci in veriler:
    fark = takipci - onceki[id]
    onceki[id] = takipci
    puan = post * 5 + yorum * 4 + begeni
    aktiflik_puanlari[(id, hafta)] = puan
    toplam_degisim[hafta - 1] += fark
    toplam_puan[hafta - 1] += puan

for i in range(8):
    print(i + 1, ". Hafta:",
          "Değişim:", toplam_degisim[i],
          "Ortalama:", round(toplam_degisim[i] / 500, 2),
          "Aktiflik:", round(toplam_puan[i] / 500, 2))

1 . Hafta: Değişim: 2783 Ortalama: 5.57 Aktiflik: 97.44
2 . Hafta: Değişim: 2231 Ortalama: 4.46 Aktiflik: 99.5
3 . Hafta: Değişim: 2679 Ortalama: 5.36 Aktiflik: 98.68
4 . Hafta: Değişim: 2938 Ortalama: 5.88 Aktiflik: 95.49
5 . Hafta: Değişim: 2594 Ortalama: 5.19 Aktiflik: 99.28
6 . Hafta: Değişim: 3273 Ortalama: 6.55 Aktiflik: 97.94
7 . Hafta: Değişim: 1786 Ortalama: 3.57 Aktiflik: 95.07
8 . Hafta: Değişim: 2807 Ortalama: 5.61 Aktiflik: 99.14


In [33]:
#NumPy İle Hesaplama:
# Takipçi sayıları
t = np.array([s[5] for s in veriler]).reshape(500, 8)
# Başlangıç takipçileri
baslangic = np.array(list(kullanici_sozlugu.values()))
# Önceki haftaların takipçi sayıları
onceki_np = np.column_stack((baslangic, t[:, :-1]))
# Takipçi farkları
fark_np = t - onceki_np
# Aktiflik puanları
post = np.array([s[2] for s in veriler]).reshape(500, 8)
begeni = np.array([s[3] for s in veriler]).reshape(500, 8)
yorum = np.array([s[4] for s in veriler]).reshape(500, 8)
puan_np = post * 5 + yorum * 4 + begeni
for i in range(8):
    print(i + 1, ". Hafta:",
          "Değişim:", np.sum(fark_np[:, i]),
          "Ortalama:", round(np.mean(fark_np[:, i]), 2),
          "Aktiflik:", round(np.mean(puan_np[:, i]), 2))

1 . Hafta: Değişim: 2783 Ortalama: 5.57 Aktiflik: 97.44
2 . Hafta: Değişim: 2231 Ortalama: 4.46 Aktiflik: 99.5
3 . Hafta: Değişim: 2679 Ortalama: 5.36 Aktiflik: 98.68
4 . Hafta: Değişim: 2938 Ortalama: 5.88 Aktiflik: 95.49
5 . Hafta: Değişim: 2594 Ortalama: 5.19 Aktiflik: 99.28
6 . Hafta: Değişim: 3273 Ortalama: 6.55 Aktiflik: 97.94
7 . Hafta: Değişim: 1786 Ortalama: 3.57 Aktiflik: 95.07
8 . Hafta: Değişim: 2807 Ortalama: 5.61 Aktiflik: 99.14


In [34]:
#Temel Python:
def python_analiz():
    onceki = kullanici_sozlugu.copy()
    degisim = [0] * 8
    puanlar = [0] * 8
    
    for id, hafta, post, begeni, yorum, takipci in veriler:
        degisim[hafta-1] += takipci - onceki[id]
        onceki[id] = takipci
        puanlar[hafta-1] += post*5 + yorum*4 + begeni

    ortalama = [sayi / 500 for sayi in degisim]
    aktiflik = [sayi / 500 for sayi in puanlar]
    return degisim, ortalama, aktiflik

In [35]:
#Numpy Hesaplama:
def numpy_analiz():
    fark = t - onceki_np
    puan = post*5 + yorum*4 + begeni
    degisim = np.sum(fark, axis=0)
    ortalama = np.mean(fark, axis=0)
    aktiflik = np.mean(puan, axis=0)

    return degisim, ortalama, aktiflik

In [36]:
#Hız TEst:
%timeit python_analiz()
%timeit numpy_analiz()

1.43 ms ± 66.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
94.4 μs ± 1.93 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [39]:
import pandas as pd
df = pd.DataFrame(veriler, columns=[
    "id", "hafta", "post", "begeni", "yorum", "takipci"])
print(df.head())

      id  hafta  post  begeni  yorum  takipci
0  U0001      1     0       5      4      110
1  U0001      2     3       0     19      139
2  U0001      3     5      25      8      170
3  U0001      4     0      41      0      202
4  U0001      5     1      13      8      237


In [40]:
#Excel Verileri Okuma:
import pandas as pd
# Temel Python (önceden okuduğumuz veriler)
print(veriler[:3])
# Pandas
df = pd.read_excel("sosyal_medya_500_kullanici_sade.xlsx",
                   sheet_name="HaftalikVeri")
df.columns = ["id", "hafta", "post", "begeni", "yorum", "takipci"]
print(df.head(3))

[('U0001', 1, 0, 5, 4, 110), ('U0001', 2, 3, 0, 19, 139), ('U0001', 3, 5, 25, 8, 170)]
      id  hafta  post  begeni  yorum  takipci
0  U0001      1     0       5      4      110
1  U0001      2     3       0     19      139
2  U0001      3     5      25      8      170


In [41]:
#Filtreleme ve Sıralama Temel Python
hafta1 = []
for satir in veriler:
    if satir[1] == 1:
        hafta1.append(satir)
sirali = sorted(hafta1, key=lambda x: x[2], reverse=True)
for satir in sirali[:5]:
    print(satir[0], satir[2])

U0009 8
U0016 8
U0056 8
U0077 8
U0080 8


In [42]:
#Fİltreleme ve Sıralama Pandas
hafta1_pd = df[df["hafta"] == 1]
sirali_pd = hafta1_pd.sort_values("post", ascending=False)
print(sirali_pd[["id", "post"]].head(5))

         id  post
2480  U0311     8
640   U0081     8
3792  U0475     8
1232  U0155     8
2576  U0323     8


In [43]:
#Haftalara Göre Gruplama: Beğeni Sayısı
#Temel Python:
def python_grup():
    toplam = [0] * 8
    for satir in veriler:
        hafta = satir[1]
        toplam[hafta - 1] += satir[3]
    return toplam
print(python_grup())

[13318, 13621, 14078, 13303, 13900, 13994, 13356, 14432]


In [44]:
#Pandas
def pandas_grup():
    return df.groupby("hafta")["begeni"].sum().tolist()
print(pandas_grup())

[13318, 13621, 14078, 13303, 13900, 13994, 13356, 14432]


In [45]:
%timeit python_grup()
%timeit pandas_grup()

423 μs ± 14.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
686 μs ± 34.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [46]:
import numpy as np

# Kullanıcı ID'leri
id_listesi = list(kullanici_sozlugu.keys())

# 0. hafta takipçi sayıları
baslangic = np.array(list(kullanici_sozlugu.values()))

# 8 haftalık veriler (500 kullanıcı x 8 hafta)
takipciler = np.array([s[5] for s in veriler]).reshape(500, 8)
postlar = np.array([s[2] for s in veriler]).reshape(500, 8)
begeniler = np.array([s[3] for s in veriler]).reshape(500, 8)
yorumlar = np.array([s[4] for s in veriler]).reshape(500, 8)

# Takipçi değişimleri ve aktiflik puanları
onceki = np.column_stack((baslangic, takipciler[:, :-1]))
farklar = takipciler - onceki
puanlar = postlar * 5 + yorumlar * 4 + begeniler

# 8 haftanın tamamını analiz et
for hafta in range(8):
    fark = farklar[:, hafta]
    puan = puanlar[:, hafta]

    print(f"\n========== {hafta + 1}. HAFTA ==========")
    print("Toplam takipçi değişimi:", np.sum(fark))
    print("Ortalama takipçi değişimi:", round(np.mean(fark), 2))
    print("Ortalama aktiflik puanı:", round(np.mean(puan), 2))

    siralamalar = {
        "En çok takipçi kazanan": np.argsort(-fark),
        "En çok takipçi kaybeden": np.argsort(fark),
        "En aktif": np.argsort(-puan),
        "En az aktif": np.argsort(puan)
    }

    for baslik, siralama in siralamalar.items():
        print("\n" + baslik + " 10 kullanıcı:")

        for i in siralama[:10]:
            deger = puan[i] if "aktif" in baslik.lower() else fark[i]
            print(id_listesi[i], ":", deger)


========== 1. HAFTA ==========
Toplam takipçi değişimi: 2783
Ortalama takipçi değişimi: 5.57
Ortalama aktiflik puanı: 97.44

En çok takipçi kazanan 10 kullanıcı:
U0125 : 35
U0437 : 35
U0213 : 35
U0445 : 35
U0409 : 34
U0255 : 34
U0496 : 34
U0361 : 34
U0457 : 34
U0093 : 34

En çok takipçi kaybeden 10 kullanıcı:
U0199 : -25
U0419 : -25
U0311 : -25
U0276 : -25
U0273 : -25
U0206 : -25
U0122 : -25
U0072 : -25
U0448 : -25
U0500 : -24

En aktif 10 kullanıcı:
U0418 : 188
U0311 : 185
U0153 : 182
U0077 : 179
U0488 : 178
U0290 : 176
U0264 : 176
U0293 : 173
U0270 : 173
U0445 : 170

En az aktif 10 kullanıcı:
U0410 : 7
U0193 : 10
U0299 : 14
U0035 : 15
U0322 : 17
U0060 : 19
U0459 : 19
U0064 : 21
U0108 : 21
U0001 : 21

========== 2. HAFTA ==========
Toplam takipçi değişimi: 2231
Ortalama takipçi değişimi: 4.46
Ortalama aktiflik puanı: 99.5

En çok takipçi kazanan 10 kullanıcı:
U0104 : 35
U0371 : 35
U0139 : 35
U0308 : 35
U0124 : 35
U0142 : 35
U0458 : 35
U0079 : 35
U0175 : 35
U0109 : 35

En çok takipçi 